# ByteRCNN — FFT-75 Scenario #1 Baseline

**Frozen benchmark**: 512-byte fragments · 75 classes · official pre-split NPZ files

Pipeline:
1. Install dependencies
2. Clone repo
3. Download `FFT-75.zip` from Google Drive
4. Unzip the archive
5. Verify dataset integrity
6. Load config (fragment_size=512)
7. Sanity training (2 epochs, 2 000 samples)
8. Full training (30 epochs, early stopping)
9. Evaluation on frozen test set
10. Zip and save outputs

> **Only one variable to change before running:**
> ```python
> FFT75_DRIVE_FILE_ID = "<YOUR_FILE_ID>"
> ```

## Cell 1 — Install Dependencies

In [ ]:
import subprocess
import sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('gdown>=5.2.0')
pip('pyyaml>=6.0')
pip('scikit-learn>=1.5')
pip('pandas>=2.2')
pip('numpy>=1.26')
pip('matplotlib>=3.9')
pip('seaborn>=0.13')
pip('tqdm>=4.66')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

## Cell 2 — Clone Repo

In [ ]:
import os
from pathlib import Path

WORKING  = Path('/kaggle/working')
REPO_DIR = WORKING / 'deepcarv'

if not REPO_DIR.exists():
    # Replace with your actual public GitHub URL:
    !git clone --depth 1 https://github.com/YOUR_USERNAME/deepcarv.git {REPO_DIR}
else:
    print('Repo already present:', REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Tell paths.py we are in Kaggle
os.environ['KAGGLE_RUNTIME'] = '1'

print('sys.path[0]:', sys.path[0])

## Cell 3 — !! Configure Google Drive ID Here !!

Get your file ID from the share URL:
```
https://drive.google.com/file/d/<FILE_ID>/view
```

In [ ]:
# ── !! ONLY CHANGE THIS !! ───────────────────────────────────────────────────
FFT75_DRIVE_FILE_ID = 'PUT_YOUR_FILE_ID_HERE'   # file ID of FFT-75.zip on Drive
ARCHIVE_NAME        = 'FFT-75.zip'               # filename to save locally
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR    = WORKING / 'data'
FFT75_DIR   = DATA_DIR / 'FFT-75'
CKPT_DIR    = WORKING / 'checkpoints'
OUTPUTS_DIR = WORKING / 'outputs'
LOGS_DIR    = WORKING / 'logs'

for d in [DATA_DIR, FFT75_DIR, CKPT_DIR, OUTPUTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Paths ready.')
print(f'  DATA_DIR    : {DATA_DIR}')
print(f'  FFT75_DIR   : {FFT75_DIR}')
print(f'  CKPT_DIR    : {CKPT_DIR}')
print(f'  OUTPUTS_DIR : {OUTPUTS_DIR}')

## Cell 4 — Download FFT-75.zip from Google Drive

In [ ]:
import gdown

archive_path = DATA_DIR / ARCHIVE_NAME

if archive_path.exists():
    print(f'Archive already downloaded: {archive_path}  ({archive_path.stat().st_size/1e6:.1f} MB)')
else:
    print(f'Downloading {ARCHIVE_NAME} from Google Drive …')
    gdown.download(
        id=FFT75_DRIVE_FILE_ID,
        output=str(archive_path),
        quiet=False,
    )
    print(f'Downloaded → {archive_path}  ({archive_path.stat().st_size/1e6:.1f} MB)')

## Cell 5 — Unzip the Archive

In [ ]:
import zipfile

archive_path = DATA_DIR / ARCHIVE_NAME

# Check if already extracted
already_extracted = (FFT75_DIR / '512' / 'train.npz').exists()

if already_extracted:
    print('Dataset already extracted.')
else:
    print(f'Extracting {archive_path} → {DATA_DIR} …')
    with zipfile.ZipFile(archive_path, 'r') as zf:
        zf.extractall(DATA_DIR)
    print('Extraction complete.')

# List what we have
print('\nContents of FFT-75/:')
for item in sorted(FFT75_DIR.rglob('*.npz')):
    size_mb = item.stat().st_size / 1e6
    print(f'  {item.relative_to(DATA_DIR)}  ({size_mb:.1f} MB)')

## Cell 6 — Verify Dataset

Checks NPZ keys, shapes, fragment lengths, and class counts.
**Stops immediately if anything is wrong.**

In [ ]:
from src.data.verify_dataset import verify_dataset

FRAGMENT_SIZE = 512   # change to 4096 to run that benchmark instead

verify_dataset(
    data_dir=FFT75_DIR,
    fragment_size=FRAGMENT_SIZE,
)
print('Dataset verification PASSED.')

## Cell 7 — Sanity Training (2 epochs · 2 000 samples)

In [ ]:
from src.training.sanity_train_bytercnn import main as sanity_main

sanity_main([
    '--data_dir',        str(FFT75_DIR),
    '--fragment_size',   str(FRAGMENT_SIZE),
    '--checkpoint_path', str(CKPT_DIR / 'sanity_bytercnn_fft75.pt'),
    '--epochs',    '2',
    '--subset',    '2000',
    '--batch_size','64',
    '--seed',      '42',
])

## Cell 8 — Full Training (30 epochs · early stopping)

In [ ]:
from src.training.train_bytercnn import main as train_main

train_main([
    '--data_dir',        str(FFT75_DIR),
    '--fragment_size',   str(FRAGMENT_SIZE),
    '--checkpoint_path', str(CKPT_DIR / 'best_bytercnn_fft75.pt'),
    '--epochs',       '30',
    '--batch_size',   '256',
    '--lr',           '1e-3',
    '--patience',     '5',
    '--grad_clip',    '1.0',
    '--seed',         '42',
])

## Cell 9 — Evaluation on Frozen Test Set

In [ ]:
from src.evaluation.evaluate_bytercnn import main as eval_main

EVAL_OUT = OUTPUTS_DIR / 'bytercnn_fft75'

eval_main([
    '--checkpoint',    str(CKPT_DIR / 'best_bytercnn_fft75.pt'),
    '--data_dir',      str(FFT75_DIR),
    '--fragment_size', str(FRAGMENT_SIZE),
    '--out_dir',       str(EVAL_OUT),
    '--batch_size',    '256',
    '--seed',          '42',
])

# Quick display
import json
with open(EVAL_OUT / 'metrics.json') as f:
    m = json.load(f)
print('\n=== Key Metrics ===')
for k in ['accuracy','macro_f1','weighted_f1','time_per_sample_ms','peak_gpu_memory_mb']:
    print(f'  {k:<35} {m[k]:.4f}')

## Cell 10 — Zip and Save Outputs

In [ ]:
import shutil

bundle_dir  = WORKING / 'ByteRCNN_baseline_bundle'
bundle_dir.mkdir(exist_ok=True)

# Copy evaluation outputs
for fname in ['metrics.json','confusion_matrix.csv','per_class_metrics.csv',
              'predictions.csv','eval_summary.txt']:
    src = EVAL_OUT / fname
    if src.exists():
        shutil.copy2(src, bundle_dir / fname)

# Copy training curve
curve = OUTPUTS_DIR / 'bytercnn_fft75' / 'training_curves.png'
if curve.exists():
    shutil.copy2(curve, bundle_dir / 'training_curves.png')

# Copy best checkpoint
best_ckpt = CKPT_DIR / 'best_bytercnn_fft75.pt'
if best_ckpt.exists():
    shutil.copy2(best_ckpt, bundle_dir / 'best_bytercnn_fft75.pt')

# Zip
zip_path = WORKING / 'ByteRCNN_baseline.zip'
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', bundle_dir)

print(f'Output bundle → {zip_path}')
print(f'Size          : {zip_path.stat().st_size / 1e6:.1f} MB')
print('\nFiles in bundle:')
for f in sorted(bundle_dir.iterdir()):
    print(f'  {f.name}')